<a href="https://colab.research.google.com/github/Thcastro2004/ECSE551-A2-ML-for-engineers/blob/main/Barnett_Cottereau_Zhang_Assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 1 - CNN

In [ ]:
#import statements
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import numpy as np
import csv
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, Subset, DataLoader
from sklearn.model_selection import train_test_split, KFold
from PIL import Image

Importing and pre-processing data using a custom Dataset for the training dataset

In [ ]:
class TrainingDataset(Dataset): #10000 samples with labels, performs random data augmentation
  def __init__(self):
    super().__init__()
    self.df = pd.read_csv('/content/drive/MyDrive/train_labels.csv')
    self.trainpath = '/content/drive/MyDrive/train/'
    self.transform = transforms.Compose([
        transforms.RandomApply(nn.ModuleList([transforms.RandomResizedCrop(size=(32,32), scale=(0.8,1))]),p=0.1),
        transforms.RandomApply(nn.ModuleList([transforms.RandomRotation((1,5))]),p=0.1),
        transforms.RandomHorizontalFlip(p=0.1),
        transforms.RandomApply(nn.ModuleList([transforms.ColorJitter((0.7,1),(0.7,1),(0.7,1),(-0.1,0.1))]),p=0.1),
        transforms.ToTensor(), #converts a PIL image to a Pytorch Tensor
        transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5])
        ])
    self.labels_dict = {
    "truck": 0,
    "deer": 1,
    "bird": 2,
    "frog": 3,
    "ship": 4,
    "horse": 5,
    "cat": 6,
    "dog": 7,
    "automobile": 8,
    "airplane": 9
}
    return

  def __getitem__(self, idx):
    img = Image.open(self.trainpath+self.df.at[idx, "id"])
    img = img.convert("RGB")
    img = self.transform(img)
    label = self.labels_dict.get(self.df.at[idx, "label"])
    return img, label

  def __len__(self):
    return len(self.df)

In [ ]:
class TestingDataset(Dataset): #10000 samples with labels, no transformations
  def __init__(self):
    super().__init__()
    self.df = pd.read_csv('/content/drive/MyDrive/train_labels.csv')
    self.trainpath = '/content/drive/MyDrive/train/'
    self.transform = transforms.Compose([
        transforms.ToTensor(), #converts a PIL image to a Pytorch Tensor
        transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5])
        ])
    self.labels_dict = {
    "truck": 0,
    "deer": 1,
    "bird": 2,
    "frog": 3,
    "ship": 4,
    "horse": 5,
    "cat": 6,
    "dog": 7,
    "automobile": 8,
    "airplane": 9
  }
    return

  def __getitem__(self, idx):
    img = Image.open(self.trainpath+self.df.at[idx, "id"])
    img = img.convert("RGB")
    img = self.transform(img)
    label = self.labels_dict.get(self.df.at[idx, "label"])
    return img, label

  def __len__(self):
    return len(self.df)

CNN Class Definition

In [ ]:
class CNN(nn.Module):

  def __init__(self):
    super().__init__()
    self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
    self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
    self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
    self.conv4 = nn.Conv2d(64, 128, 3, padding=1)
    self.conv5 = nn.Conv2d(128, 256, 4, padding=1)
    self.conv6 = nn.Conv2d(256, 512, 4, padding=1)
    self.pool = nn.MaxPool2d(2, 2)
    self.lin1 = nn.Linear(8192, 4096)
    self.lin2 = nn.Linear(4096, 512)
    self.lin3 = nn.Linear(512, 128)
    self.lin4 = nn.Linear(128, 10)
    self.batchnorm1 = nn.BatchNorm2d(16)
    self.batchnorm2 = nn.BatchNorm2d(32)
    self.batchnorm3 = nn.BatchNorm2d(64)
    self.batchnorm4 = nn.BatchNorm2d(128)
    self.batchnorm5 = nn.BatchNorm2d(256)
    self.batchnorm6 = nn.BatchNorm2d(512)
    self.dropout = nn.Dropout(0.3)
    return

  def forward(self, x):
    x = nn.functional.relu(self.batchnorm1(self.conv1(x)))
    x = nn.functional.relu(self.batchnorm2(self.conv2(x)))
    x = self.pool(x)
    x = nn.functional.relu(self.batchnorm3(self.conv3(x)))
    x = nn.functional.relu(self.batchnorm4(self.conv4(x)))
    x = self.pool(x)
    # x = nn.functional.relu(self.batchnorm5(self.conv5(x)))
    # x = nn.functional.relu(self.batchnorm6(self.conv6(x)))
    # x = self.pool(x)
    x = torch.flatten(x, 1)
    x = nn.functional.relu(self.lin1(x))
    x = self.dropout(x)
    x = nn.functional.relu(self.lin2(x))
    x = self.dropout(x)
    x = nn.functional.relu(self.lin3(x))
    x = self.dropout(x)
    x = self.lin4(x)
    return x

CNN Training

In [ ]:
#split training data
training_dataset = TrainingDataset() #initiate Dataset
testing_dataset = TestingDataset()

indices = list(range(len(training_dataset)))
train_indices, test_indices = train_test_split(indices, test_size=0.3, random_state=42)
train_dataset = Subset(training_dataset, train_indices)
validation_dataset = Subset(testing_dataset, train_indices)
final_test_dataset = Subset(testing_dataset, test_indices) #for final model performance analysis

results = []
weights = []
kf = KFold(shuffle=True, random_state=42)

for i, (train_idx, test_idx) in enumerate(kf.split(train_indices)):
  cnn = CNN()
  cnn.train()
  loss_function = nn.CrossEntropyLoss()
  optimizer = torch.optim.Adam(cnn.parameters(), 0.0003, weight_decay=0.0001)

  kf_train_dataset = Subset(train_dataset, train_idx)
  kf_train_dataloader = DataLoader(kf_train_dataset, batch_size=64, shuffle=True, num_workers=2)
  kf_test_dataset = Subset(validation_dataset, test_idx)
  kf_test_dataloader = DataLoader(kf_test_dataset, batch_size=64, shuffle=False, num_workers=2)

  training_loss = []
  for epoch in range(5):
    for input, label in kf_train_dataloader:
      output = cnn(input)
      loss = loss_function(output, label)
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()
      training_loss.append(loss.item())
      print("batch loss:"+str(loss.item()))
    print(f"epoch {epoch}: {loss.item()}")

  plt.plot(training_loss)
  plt.xlabel("Batch number")
  plt.ylabel("Loss")
  plt.title("Batch-wise Training Loss")
  plt.show()
  fold_filename = f"/content/drive/MyDrive/cnn_fold_{i+1}.pth"
  torch.save(cnn.state_dict(), fold_filename)

  cnn.eval()
  with torch.no_grad():
    correct = 0
    for input, label in kf_test_dataloader:
      output = cnn(input)
      prediction = output.argmax(dim=1)
      correct += (prediction == label).sum().item()
  results.append(correct/len(test_idx))
  print(results)

batch loss:2.3032054901123047
batch loss:2.37849497795105
batch loss:2.3575639724731445
batch loss:2.337953805923462
batch loss:2.415170431137085
batch loss:2.3691964149475098


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


CNN Performance Evaluation

Creating a Dataset for the 2000 unlabelled images for Kaggle

In [ ]:
class Test551(Dataset): #2000

  def __init__(self):
    super().__init__()
    self.df = pd.read_csv('/content/drive/MyDrive/sample_submission.csv')
    self.trainpath = '/content/drive/MyDrive/test/'
    self.transform = transforms.ToTensor() #converts a PIL image to a Pytorch Tensor
    self.labels_dict = {
    "truck": 0,
    "deer": 1,
    "bird": 2,
    "frog": 3,
    "ship": 4,
    "horse": 5,
    "cat": 6,
    "dog": 7,
    "automobile": 8,
    "airplane": 9
    }
    return

  def __getitem__(self, idx):
    img = Image.open(self.trainpath+self.df.at[idx, "id"])
    img = img.convert("RGB")
    img = self.transform(img)
    label = 0 #no labels assigned since this is the test set
    return img, label

  def __len__(self):
    return len(self.df)

CNN Prediction for Kaggle

In [ ]:

kaggle_dataset = Test551()
kaggle_dataloader = DataLoader(kaggle_dataset, batch_size=64, shuffle=True, num_workers=2)


for input, label in kaggle_dataloader:
  output =



In [ ]:
conv1 = nn.Conv2d(3, 16, 3, padding=1)
conv2 = nn.Conv2d(16, 32, 3, padding=1)
pool = nn.MaxPool2d(2, 2)
conv3 = nn.Conv2d(32, 64, 4, padding=1)
conv4 = nn.Conv2d(64, 128, 4, padding=1)
conv5 = nn.Conv2d(128, 256, 3, padding=1)
conv6 = nn.Conv2d(256, 512, 3, padding=1)
lin1 = nn.Linear(8192, 512)
lin2 = nn.Linear(512, 128)
lin3 = nn.Linear(128, 10)
batchnorm1 = nn.BatchNorm2d(16)
batchnorm2 = nn.BatchNorm2d(32)
batchnorm3 = nn.BatchNorm2d(64)
batchnorm4 = nn.BatchNorm2d(128)
batchnorm5 = nn.BatchNorm2d(256)
batchnorm6 = nn.BatchNorm2d(512)

x = torch.randn(1, 3, 32, 32)
x = nn.functional.relu(batchnorm1(conv1(x)))
x = nn.functional.relu(batchnorm2(conv2(x)))
x = pool(x)
x = nn.functional.relu(batchnorm3(conv3(x)))
x = nn.functional.relu(batchnorm4(conv4(x)))
x = pool(x)
# x = nn.functional.relu(conv5(x))
# x = nn.functional.relu(conv6(x))
# x = pool(x)
x = torch.flatten(x, 1)

flattened_size = x.shape[1]
print(flattened_size)
# x = nn.functional.relu(lin1(x))
# x = nn.functional.relu(lin2(x))
# x = lin3(x)


6272


# Task 1 - Vision Transformer

In [ ]:
dataset = Dataset551()


# Task 2 - Free for All